# OCR Bilans Fiscaux Algériens — V16 — JSON contrôlé + Liasse Excel annotée

> 1 PDF → 1 JSON contrôlé → 1 classeur Excel annoté. Intègre TOUS les correctifs v8→v16.

## Journal
| Version | Changement |
|---|---|
| V10C | Classification par signature texte (anti 3/↔9/, 5/↔9/, 6/↔8/) |
| V11A | Identité (NIF15, millésimes, concordance), moteur cohérence, transposition Excel |
| V13 | Arrondi ±1 DA en jaune, écart significatif en rouge, traçabilité PDF+page, chemins /mnt/Risk/Model |
| V14 | Annexes structurées + sous-totaux, TCR complet (rabais obtenus, reprises), jamais écraser formules |
| V16 | Sous-totaux ACTIF (corporelles, financières, créances, disponibilités) en formules contrôlées ; garde-fou AUTRE ; progression live |

## Règle d'or
Ne jamais inventer de valeur : absent/illisible → null.

In [ ]:
%pip install -q -U 'transformers>=4.57.0' accelerate pymupdf pillow psutil
print('✅ Dépendances OK')

In [ ]:
import time, json, re, gc, copy, unicodedata, shutil, subprocess, sys
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import Counter
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import openpyxl
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Border, Font, PatternFill, Protection, Side
print('✅ Imports OK')

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS = 4096
IMAGE_MAX_SIZE = 2024
MIN_PIXELS = 4*32*32
MAX_PIXELS = 2000*32*32
PDF_ZOOM = 3.0
BLANK_THRESHOLD = 0.95
CLASSIF_BATCH_SIZE = 16
GPU_BATCH_SIZE = 4
INPUT_DIR = Path('/mnt/Risk/bilans_in')
OUTPUT_DIR = Path('/mnt/Risk/bilans_out')
JSON_DIR = OUTPUT_DIR / 'json_bilans_v16'
CONTROLE_DIR = OUTPUT_DIR / 'json_controles_v16'
XLSX_DIR = OUTPUT_DIR / 'liasses_xlsx_v16'
LOG_PATH = OUTPUT_DIR / 'pipeline_bilans_v16.log'
MODEL_DIR = Path('/mnt/Risk/Model')
MODELE_XLSX = MODEL_DIR / 'Liasse_fiscale_G2_BNP_Paribas_El_Djazair.xlsx'
RECALC_TOOL = MODEL_DIR / 'recalc_liasse.py'
for d in (OUTPUT_DIR, JSON_DIR, CONTROLE_DIR, XLSX_DIR): d.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
RECALC_DISPONIBLE = Path(RECALC_TOOL).exists() and bool(shutil.which('soffice'))
print('Device:', DEVICE, '| PDF:', len(pdfs), '| Recalc:', RECALC_DISPONIBLE)

In [ ]:
def log(msg):
    ligne = datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ' — ' + str(msg)
    print(ligne, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as f: f.write(ligne + chr(10))
print('✅ Log OK')

In [ ]:
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'left'
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True, low_cpu_mem_usage=True, quantization_config=FP8Config(dequantize=True))
model.eval()
print('✅ Modèle chargé en ' + str(round(time.time()-t0,1)) + 's')

In [ ]:
def chunks(lst, n):
    for i in range(0, len(lst), n): yield lst[i:i+n]
def resize(img, max_side=IMAGE_MAX_SIZE):
    w,h = img.size
    if max(w,h) <= max_side: return img
    r = max_side/max(w,h); return img.resize((int(w*r), int(h*r)), Image.LANCZOS)
def strip_accents(s): return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))
def norm_key(s): return re.sub('[^a-z0-9]+','_', strip_accents(s).lower()).strip('_')
def estimate_skew(img):
    small = img.convert('L').copy(); small.thumbnail((500,500))
    def score(a):
        r = np.array(small.rotate(a, expand=True, fillcolor=255)) < 128
        return float(((r.sum(axis=1))**2).sum())
    best = max(range(-12,13,2), key=score)
    best = max([best-1,best-0.5,best,best+0.5,best+1], key=score)
    return best if abs(best) >= 1 else 0.0
def deskew(img):
    a = estimate_skew(img)
    if a: img = img.rotate(a, expand=True, fillcolor=(255,255,255), resample=Image.BICUBIC)
    return img
def is_blank(image, threshold=BLANK_THRESHOLD):
    arr = np.array(image.convert('L')); return (arr > 240).sum()/arr.size >= threshold
def pdf_to_pages(path, zoom=PDF_ZOOM):
    doc = fitz.open(path); matrix = fitz.Matrix(zoom, zoom); pages = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
        pages.append({'index': i, 'image': resize(deskew(img))})
    doc.close(); return pages
def parse_json(text):
    try:
        m = re.search(r'\{.*\}', text or '', re.S); return json.loads(m.group()) if m else {}
    except Exception: return {}
def apply_template(messages):
    try: return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError: return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
def _decode(o, n): return processor.decode(o[n:], skip_special_tokens=True, clean_up_tokenization_spaces=False)
def ask_single(prompt, image):
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    inputs = processor(text=[apply_template(msgs)], images=[image], return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return {'text': _decode(out[0], inputs['input_ids'].shape[1]), 'tokens_in': int(inputs['input_ids'].shape[1]), 'tokens_out': int(out[0].shape[0]-inputs['input_ids'].shape[1]), 'elapsed': round(time.time()-t0,2)}
def ask_batch(prompt, images):
    if not images: return []
    if len(images) == 1: return [ask_single(prompt, images[0])]
    msgs = [[{'role':'user','content':[{'type':'image','image':im},{'type':'text','text':prompt}]}] for im in images]
    inputs = processor(text=[apply_template(m) for m in msgs], images=images, return_tensors='pt', padding=True).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    el = time.time()-t0; n = inputs['input_ids'].shape[1]; attn = inputs.get('attention_mask')
    return [{'text': _decode(out[i], n), 'tokens_in': int(attn[i].sum().item()) if attn is not None else n, 'tokens_out': int(out[i].shape[0]-n), 'elapsed': round(el/len(images),2)} for i in range(len(images))]
print('✅ Utilitaires OK')

In [ ]:
# SCHEMAS V16 : ACTIF avec sous-totaux contrôlés ; TCR avec rabais obtenus + reprises
SCHEMAS = {
 'ACTIF': {'cols': ['montant_brut','amortissements_provisions_pertes','net_n','net_n1'], 'postes': {
  'ecarts_acquisition_goodwill':'Ecart d acquisition goodwill','immobilisations_incorporelles':'Immobilisations incorporelles',
  'immobilisations_corporelles':'Immobilisations corporelles','terrains':'Terrains','batiments':'Batiments',
  'autres_immobilisations_corporelles':'Autres Immobilisations corporelles','immobilisations_en_concession':'Immobilisations en concession',
  'immobilisations_en_cours':'Immobilisations en cours','immobilisations_financieres':'Immobilisations financières',
  'titres_mis_en_equivalence':'Titres mis en equivalence','autres_participations_creances':'Autres participations et creances rattachees',
  'autres_titres_immobilises':'Autres titres immobilises','prets_actifs_financiers_non_courants':'Prets et autres actifs financiers non courants',
  'impots_differes_actif':'Impots Differes Actif','total_actif_non_courant':'TOTAL ACTIF NON COURANT',
  'stocks_encours':'Stocks et encours','creances_et_emplois_assimiles':'Creances et emplois assimiles',
  'clients':'Clients','autres_debiteurs':'Autres debiteurs','impots_assimiles_actif':'Impots et assimiles',
  'autres_creances_assimiles':'Autres Creances et Emplois assimiles','disponibilites_et_assimiles':'Disponibilites et assimiles',
  'placements_financiers_courants':'Placements et autres actifs financiers courants','tresorerie_actif':'Tresorerie',
  'total_actif_courant':'TOTAL ACTIF COURANT','total_general_actif':'TOTAL GENERAL ACTIF'}},
 'PASSIF': {'cols': ['n','n1'], 'postes': {
  'capital_emis':'Capital emis','capital_non_appele':'Capital non appele','primes_reserves':'Primes et reserves',
  'ecart_reevaluation':'Ecart de reevaluation','ecart_equivalence':'Ecart d equivalence','resultat_net_passif':'Resultat net',
  'report_a_nouveau':'Report a nouveau','part_societe_consolidante':'Part de la societe consolidante','part_minoritaires':'Part des minoritaires',
  'total_capitaux_propres':'TOTAL I','emprunts_dettes_financieres':'Emprunts et dettes financieres',
  'impots_differes_provisionnes':'Impots differes et provisionnes','autres_dettes_non_courantes':'Autres dettes non courantes',
  'provisions_produits_avance':'Provisions et produits constatés d avance','total_passifs_non_courants':'TOTAL PASSIFS NON COURANTS II',
  'fournisseurs_rattaches':'Fournisseurs et comptes rattaches','impots_passif':'Impots','autres_dettes':'Autres dettes',
  'tresorerie_passif':'Tresorerie Passif','total_passifs_courants':'TOTAL PASSIFS COURANTS','total_general_passif':'TOTAL GENERAL PASSIF'}},
 'TCR': {'cols': ['n_debit','n_credit','n1_debit','n1_credit'], 'postes': {
  'ventes_marchandises':'Ventes de Marchandises','produits_fabriques':'Produits Fabriques','prestations_services':'Prestations de Services',
  'ventes_travaux':'Ventes de Travaux','produits_annexes':'Produits Annexes','rabais_remises_ristournes_accordes':'Rabais remises ristournes accordes',
  'chiffre_affaires_net':'Chiffre d affaires net','production_stockee_destockee':'Production Stockee ou destockee',
  'production_immobilisee':'Production immobilisee','subvention_exploitation':'Subvention d exploitation','production_exercice':'I-Production de l exercice',
  'achats_marchandises_vendues':'Achats de Marchandises vendues','matieres_premieres':'Matieres premieres','autres_approvisionnements':'Autres Approvisionnements',
  'variation_stocks':'Variation des Stocks','achats_etudes_prestations':'Achats d Etudes et de Prestations de services','autres_consommations':'Autres consommations',
  'rabais_remises_obtenus_achats':'Rabais remises ristournes obtenus sur achats','sous_traitance_generale':'Sous-traitance generale','locations':'Locations',
  'entretien_reparations':'Entretien reparations et maintenance','primes_assurances':'Primes d assurances','personnel_exterieur':'Personnel exterieur a l entreprise',
  'remuneration_intermediaires':'Remuneration d intermediaires et honoraires','publicite':'Publicite','deplacements_missions':'Deplacements missions et receptions',
  'autres_services':'Autres services','rabais_remises_obtenus_services':'Rabais remises ristournes obtenus sur services exterieurs',
  'consommations_exercice':'II-Consommations de l exercice','valeur_ajoutee_exploitation':'III-Valeur ajoutee d exploitation',
  'charges_personnel':'Charges de personnel','impots_taxes_assimiles':'Impots et taxes et versements assimiles','excedent_brut_exploitation':'IV-Excedent brut d exploitation',
  'autres_produits_operationnels':'Autres produits operationnels','autres_charges_operationnelles':'Autres charges operationnelles',
  'dotations_amortissements':'Dotations aux amortissements','provisions':'Provisions','pertes_valeur':'Perte de Valeur',
  'reprises_pertes_valeur_provisions':'Reprise sur pertes de valeur et provisions','resultat_operationnel':'V-Resultat operationnel',
  'produits_financiers':'Produits financiers','charges_financieres':'Charges financieres','resultat_financier':'VI-Resultat Financier',
  'resultat_ordinaire':'VII-Resultat ordinaire','elements_extraordinaires_produits':'Elements extraordinaires Produits',
  'elements_extraordinaires_charges':'Elements extraordinaires Charges','resultat_extraordinaire':'VIII-Resultat extraordinaire',
  'impots_exigibles_resultats':'Impots exigibles sur resultats','impots_differes_resultats':'Impots differes sur resultats','resultat_net_exercice':'RESULTAT NET DE L EXERCICE'}},
 'DECL': {'cols': ['valeur'], 'kinds': {'nif':'nif','raison_sociale':'texte','activite_principale':'texte','registre_commerce':'texte','adresse_siege':'texte','cac_cabinet':'texte','cac_nom':'texte','exercice_annee':'annee','annee_souscription':'annee'}, 'postes': {
  'nif':'NIF','raison_sociale':'Designation','activite_principale':'Activite','registre_commerce':'Registre Commerce',
  'adresse_siege':'Adresse','cac_cabinet':'Cabinet CAC','cac_nom':'Nom CAC','exercice_annee':'Exercice Annee',
  'annee_souscription':'Annee souscription','chiffre_affaires_global_ht':'CA global HT','resultat_comptable':'Resultat comptable','resultat_fiscal':'Resultat fiscal'}},
 'A1': {'cols': ['solde_debut','debit','credit','solde_fin'], 'postes': {'stocks_marchandises':'Stocks de marchandises','matieres_fournitures':'Matieres et fournitures','autres_approvisionnements':'Autres approvisionnements','encours_production_biens':'Encours production biens','encours_production_services':'Encours production services','stocks_produits':'Stocks de produits','stocks_provenant_immobilisations':'Stocks provenant immobilisations','stocks_exterieur':'Stocks a exterieur','total':'TOTAL'}},
 'A3': {'cols': ['montant'], 'postes': {'charges_locatives':'Charges locatives','etudes_recherches':'Etudes et recherches','documentation_divers':'Documentation et divers','transports_biens':'Transports de biens','frais_postaux':'Frais postaux','services_bancaires':'Services bancaires','cotisations_divers':'Cotisations et divers','total_autres_services':'TOTAL (1)','remunerations_personnel':'Remunerations du personnel','remuneration_exploitant':'Remuneration exploitant','cotisations_sociales':'Cotisations organismes sociaux','charges_sociales_exploitant':'Charges sociales exploitant','autres_charges_sociales':'Autres charges sociales','autres_charges_personnel':'Autres charges personnels','total_charges_personnel':'TOTAL (2)','impots_sur_remunerations':'Impots sur remunerations','impots_non_recuperables':'Impots non recuperables','autres_impots_taxes':'Autres impots et taxes','total_impots':'TOTAL (3)','total_general':'TOTAL (1)+(2)+(3)'}},
 'A4': {'cols': ['montant'], 'postes': {'redevances_concessions_charges':'Redevances concessions (charges)','moins_values_sorties_actifs':'Moins values sorties actifs','jetons_presence_charges':'Jetons presence (charges)','pertes_creances_irrecouvrables':'Perte creances irrecouvrables','quote_part_operations_commun_charges':'Quote-part operations commun (charges)','amendes_penalites_dons':'Amendes penalites dons','charges_exceptionnelles_gestion':'Charges exceptionnelles gestion','autres_charges_gestion':'Autres charges gestion','total_charges':'TOTAL charges','redevances_concessions_produits':'Redevances concessions (produits)','plus_values_sorties_actifs':'Plus values sorties actifs','jetons_presence_produits':'Jetons presence (produits)','quotes_parts_subventions_virees':'Quotes-parts subventions virees','quote_part_operations_commun_produits':'Quote-part operations commun (produits)','rentrees_creances_amorties':'Rentree creances amorties','produits_exceptionnels_gestion':'Produits exceptionnels gestion','autres_produits_gestion':'Autres produits gestion','total_produits':'TOTAL produits'}},
 'A5': {'cols': ['dotations_cumulees_debut','dotations_exercice','diminutions_elements_sortis','dotations_cumulees_fin','dotations_fiscales_exercice','ecarts'], 'postes': {'goodwill':'Goodwill','immobilisations_incorporelles':'Immobilisations incorporelles','immobilisations_corporelles':'Immobilisations corporelles','participations':'Participations','autres_actifs_financiers_non_courants':'Autres actifs financiers non courants','total':'TOTAL'}},
 'A6': {'cols': ['montants_bruts','tva_deduite','montant_net_a_amortir'], 'postes': {'goodwill':'Goodwill','immobilisations_incorporelles':'Immobilisations incorporelles','immobilisations_corporelles':'Immobilisations corporelles','participations':'Participations','autres_actifs_financiers_non_courants':'Autres actifs financiers non courants','total':'TOTAL'}},
 'A7': {'dynamic': True, 'str_cols': ['date_acquisition'], 'cols': ['date_acquisition','montant_net_actif','amortissements_pratiques','valeur_nette_comptable','prix_cession','plus_value','moins_value'], 'postes': {}},
 'A8': {'cols': ['provisions_cumulees_debut','dotations_exercice','reprises_exercice','provisions_cumulees_fin'], 'postes': {'pertes_valeur_stocks':'Pertes valeurs stocks','pertes_valeur_creances':'Pertes valeurs creances','pertes_valeur_actions':'Pertes valeurs actions','provisions_pensions':'Provisions pensions','provisions_litiges':'Provisions litiges','autres_provisions_personnel':'Autres provisions personnel','provisions_impots':'Provisions impots','autres_provisions':'Autres provisions','total':'TOTAL'}},
 'A81': {'dynamic': True, 'cols': ['valeur_creance','perte_valeur_constituee'], 'postes': {}},
 'A82': {'dynamic': True, 'cols': ['valeur_nominale_debut','perte_valeur_constituee','valeur_nette_comptable'], 'postes': {}},
 'A9': {'cols': ['montant'], 'postes': {'resultat_net_benefice':'Resultat net Benefice','resultat_net_perte':'Resultat net Perte','charges_immeubles_non_affectes':'Charges immeubles non affectes','quote_part_cadeaux_publicitaires':'Quote-part cadeaux publicitaires','quote_part_sponsoring':'Quote-part sponsoring','frais_reception':'Frais reception','cotisations_dons':'Cotisations dons','impots_taxes_non_deductibles':'Impots taxes non deductibles','provisions_non_deductibles':'Provisions non deductibles','amortissements_non_deductibles':'Amortissements non deductibles','quote_part_frais_rd':'Quote-part frais RD','amortissements_credit_bail_preneur':'Amortissements credit bail preneur','loyers_hors_produits_financiers_bailleur':'Loyers hors produits financiers bailleur','ibs_impot_exigible':'IBS impot exigible','ibs_impot_differe':'IBS impot differe','pertes_valeur_non_deductibles':'Pertes valeurs non deductibles','amendes_penalites':'Amendes penalites','autres_reintegrations':'Autres reintegrations','total_reintegrations':'Total reintegrations','plus_values_cession_actif_immobilise':'Plus values cession actif','produits_plus_values_actions_bourse':'Produits plus values actions','revenus_distribution_benefices':'Revenus distribution benefices','amortissements_credit_bail_bailleur':'Amortissements credit bail bailleur','loyers_hors_charges_financieres_preneur':'Loyers hors charges financieres preneur','complement_amortissements':'Complement amortissements','autres_deductions':'Autres deductions','total_deductions':'Total deductions','total_deficits_a_deduire':'Total deficits a deduire','resultat_fiscal_benefice':'Resultat fiscal Benefice','resultat_fiscal_deficit':'Resultat fiscal Deficit'}},
 'A10': {'cols': ['montant'], 'postes': {'origine_report_a_nouveau_n1':'Report a nouveau N-1','origine_resultat_n1':'Resultat N-1','origine_prelevements_reserves':'Prelevements reserves','origine_total':'TOTAL origine','affectation_reserves':'Reserves','affectation_augmentation_capital':'Augmentation capital','affectation_dividendes':'Dividendes','affectation_report_a_nouveau':'Report a nouveau','affectation_total':'TOTAL affectation'}},
 'A11': {'dynamic': True, 'cols': ['capitaux_propres','dont_capital','quote_part_capital_pct','resultat_dernier_exercice','prets_avances','dividendes_encaisses','valeur_comptable_titres'], 'postes': {}},
 'A12': {'dynamic': True, 'str_cols': ['nif','adresse'], 'cols': ['nif','adresse','montant_percu'], 'postes': {}},
 'A13': {'dynamic': True, 'cols': ['ca_imposable','ca_exonere','tap_acquittee'], 'postes': {}},
 'ST': {'dynamic': True, 'str_cols': ['nif','article','adresse','reference_contrat'], 'cols': ['nif','article','adresse','reference_contrat','montant'], 'postes': {}},
 'REM': {'dynamic': True, 'str_cols': ['annee_versement'], 'cols': ['nombre_parts','annee_versement','traitement_emoluments','representation_mission_forfait','representation_mission_remboursement','frais_pro_forfait','frais_pro_remboursement'], 'postes': {}},
 'DIST': {'cols': ['montant'], 'postes': {'montant_global_brut':'Montant global brut','paye_par_societe':'Paye par societe','paye_par_etablissement':'Paye par etablissement','total_revenus_repartis':'Total revenus repartis'}}
}
ANNEXE_NUM_MAP = {1:'A1',2:'A2',3:'A3',4:'A4',5:'A5',6:'A6',7:'A7',8:'A8',9:'A9',10:'A10',11:'A11',12:'A12',13:'A13'}
print('✅ Schémas V16 OK —', len(SCHEMAS), 'tableaux')

In [ ]:
# CLASSIFICATION PAR SIGNATURE TEXTE (V10C) — jamais par ordre
TITLE_SIGNATURES = [
 ('A81',['RELEVE DES PERTES DE VALEURS SUR CREANC'],[]), ('A82',['RELEVE DES PERTES DE VALEURS SUR ACTIONS'],[]),
 ('A1',['MOUVEMENTS DES STOCKS'],[]), ('A2',['FLUCTUATION DE LA PRODUCTION STOCKEE'],[]),
 ('A3',['CHARGES DE PERSONNEL','VERSEMENTS ASSIMILES'],[]), ('A4',['AUTRES CHARGES ET PRODUITS OPERATIONNELS'],[]),
 ('A5',['AMORTISSEMENTS ET PERTES DE VALEURS'],[]), ('A6',['IMMOBILISATIONS CREEES OU ACQUISES'],[]),
 ('A7',['IMMOBILISATIONS CEDEES'],[]), ('A8',['PROVISIONS ET PERTES DE VALEURS'],['RELEVE DES PERTES']),
 ('A9',['DETERMINATION DU RESULTAT FISCAL'],[]), ('A10',['AFFECTATION DU RESULTAT ET DES RESERVES'],[]),
 ('A11',['TABLEAU DES PARTICIPATIONS'],[]), ('A12',['COMMISSIONS ET COURTAGES'],[]),
 ('A13',['TAXE SUR L ACTIVITE PROFESSIONNELLE'],[]), ('ST',['OPERATIONS DE SOUS-TRAITANCE'],['COMMISSIONS ET COURTAGES']),
 ('REM',['REMUNERATIONS VERSEES AUX MEMBRES'],[]), ('DIST',['REPARTITION DES PRODUITS DES ACTIONS'],[])]
MAIN_SIGNATURES = [('ACTIF',['BILAN','ACTIF'],['PASSIF']), ('PASSIF',['BILAN','PASSIF'],[]), ('TCR',['COMPTE DE RESULTAT'],[]), ('DECL',['DECLARATION'],[])]
def _norm(t):
    t = strip_accents(t or '').upper(); t = t.replace(chr(39),' ').replace(chr(8217),' ')
    return ' '.join(re.sub(r'[^A-Z0-9/ ]+',' ', t).split())
_TITLE_SIGS = [(c,[_norm(m) for m in ms],[_norm(f) for f in fs]) for c,ms,fs in TITLE_SIGNATURES]
_MAIN_SIGS = [(c,[_norm(m) for m in ms],[_norm(f) for f in fs]) for c,ms,fs in MAIN_SIGNATURES]
def _match(t, sigs):
    out = []
    for c, musts, forb in sigs:
        if all(m in t for m in musts) and not any(f in t for f in forb):
            if c not in out: out.append(c)
    return out
def _num_seg(t):
    codes, nums = [], []
    for m in re.finditer(r'(?:^| )(1[0-3]|[0-9])/([0-9])?(?![0-9])', t):
        n = int(m.group(1)); s = m.group(2)
        c = ('A8'+s) if (n==8 and s in ('1','2')) else ANNEXE_NUM_MAP.get(n)
        if c and c not in codes: codes.append(c); nums.append(n)
    return codes, nums
def types_from_title(titre):
    brut = _norm(titre)
    if not brut: return [], []
    segs = [_norm(s) for s in re.split(r'\s\|\s', titre or '') if s.strip()] or [brut]
    types, nums = [], []
    for seg in segs:
        tx = _match(seg, _TITLE_SIGS)
        if tx:
            for c in tx:
                if c not in types: types.append(c)
        else:
            cs, ns = _num_seg(seg)
            for c in cs:
                if c not in types: types.append(c)
            nums += ns
    if ('A81' in types or 'A82' in types) and 'A8' in types and not any('PROVISIONS ET PERTES DE VALEURS' in s for s in segs): types.remove('A8')
    if not types: types = _match(brut, _MAIN_SIGS)
    if 'ACTIF' in types and 'PASSIF' in types: types = ['ACTIF'] if brut.find('ACTIF') < brut.find('PASSIF') else ['PASSIF']
    return types, nums
RULES = ['REGLES: JSON valide uniquement, sans markdown, sans backticks.','Aucune valeur inventee. Absent/illisible: null.','Montants en nombres JSON sans separateurs.','Montant entre parentheses = negatif.','Textes (nif, adresses, noms, dates) en chaines telles quelles.']
def build_prompt(types):
    L = ['Lis cette page scannee d une liasse fiscale algerienne (imprime Serie G).','Extrais en JSON strict les tableaux suivants.']
    L.append('Structure: { type_page, numero_page_imprimee, titre_page, entete: {entreprise, nif, exercice, exercice_du, exercice_au, adresse, activite, serie_g}, puis une cle par code.}')
    L.append('EN-TETE obligatoire sur CHAQUE page: nif (15 chiffres colles), entreprise (forme juridique comprise), exercice (jj/mm/aaaa).')
    for t in types:
        spec = SCHEMAS[t]; L.append('--- Code ' + t + ' : ' + (spec.get('titre') or t) + ' ---')
        if spec.get('dynamic'):
            L.append('Tableau libre: {lignes: [ {libelle_imprime, valeurs: {colonne: nombre ou texte ou null}} ]}'); L.append('Colonnes: ' + ', '.join(spec['cols']))
        else:
            L.append('Retourne: {lignes: [ {row_code, libelle_imprime, valeurs: {colonne: nombre ou null}} ]}'); L.append('Colonnes: ' + ', '.join(spec['cols']))
            L.append('Row_code autorises:')
            for k, lab in spec['postes'].items(): L.append('- ' + k + ' : ' + lab)
            L.append('Retourne seulement les row_code avec au moins une valeur non nulle.')
    L += RULES
    return chr(10).join(L)
PROMPT_CLASSIF = ('Lis cette page scannee d une liasse Serie G. Recopie les TITRES DE TABLEAUX imprimes, separes par |. '
 'Reponds en JSON strict: {"titre": ..., "type": ACTIF|PASSIF|TCR|DECL|ANNEXE|AUTRE}. '
 'Ne recopie pas les libelles de lignes ni les en-tetes de colonnes.')
VALID_CODES = set(SCHEMAS.keys()) | {'AUTRE','BLANCHE'}
print('✅ Classification V16 OK')

In [ ]:
# NORMALISATION
def norm_str(v):
    if v is None or isinstance(v, bool): return None
    s = ' '.join(str(v).split()); return s if s and s.lower() not in ('null','none','n/a','na','-') else None
def norm_montant(v):
    if v is None or isinstance(v, bool): return None
    if isinstance(v,(int,float)): return float(v)
    s = str(v).strip(); neg = (s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s = re.sub('[^0-9.,-]','',s)
    if not s: return None
    if s.count(',')==1 and '.' not in s: s = s.replace(',','.')
    elif ',' in s: s = s.replace(',','')
    elif s.count('.')>1: s = s.replace('.','')
    try: return -float(s) if neg else float(s)
    except Exception: return None
def norm_nif(v):
    s = norm_str(v); return re.sub('[^0-9]','',s) if s else None
def norm_annee4(v):
    m = re.findall(r'20[0-9]{2}', norm_str(v) or ''); return m[-1] if m else None
def norm_by_kind(v, kind):
    return {'texte': norm_str, 'nif': norm_nif, 'annee': norm_annee4}.get(kind, norm_montant)(v)
def normalise_fixed(t, block):
    spec = SCHEMAS[t]; kinds = spec.get('kinds', {})
    lignes = (block or {}).get('lignes') or []
    l2k = {norm_key(v): k for k, v in spec['postes'].items()}
    row_map = {}
    for lg in lignes:
        if not isinstance(lg, dict): continue
        rc = norm_str(lg.get('row_code'))
        if rc not in spec['postes']:
            lab = norm_str(lg.get('libelle_imprime'))
            if lab and norm_key(lab) in l2k: rc = l2k[norm_key(lab)]
        if rc in spec['postes']: row_map[rc] = lg
    out = {}
    for key in spec['postes']:
        vals = (row_map.get(key) or {}).get('valeurs') or {}
        if not isinstance(vals, dict): vals = {}
        out[key] = {col: norm_by_kind(vals.get(col), kinds.get(key,'montant')) for col in spec['cols']}
    return out
def normalise_dynamic(t, block):
    spec = SCHEMAS[t]; str_cols = set(spec.get('str_cols', [])); out = []
    for lg in (block or {}).get('lignes') or []:
        if not isinstance(lg, dict): continue
        vals = lg.get('valeurs') or {}
        if not isinstance(vals, dict): vals = {}
        row = {'libelle_imprime': norm_str(lg.get('libelle_imprime'))}
        for col in spec['cols']: row[col] = norm_str(vals.get(col)) if col in str_cols else norm_montant(vals.get(col))
        if row['libelle_imprime'] or any(v is not None for k,v in row.items() if k!='libelle_imprime'): out.append(row)
    return out
def normalise_table(t, block):
    if t == 'AUTRE': return block or {}
    return normalise_dynamic(t, block) if SCHEMAS[t].get('dynamic') else normalise_fixed(t, block)
def normalise_entete(data):
    ent = data.get('entete') or {}
    if not isinstance(ent, dict): ent = {}
    return {k: (norm_nif(ent.get('nif')) if k=='nif' else norm_str(ent.get(k))) for k in ('entreprise','nif','exercice','exercice_du','exercice_au','adresse','activite','serie_g')}
def count_values(obj):
    c = 0
    if isinstance(obj, dict):
        for k,v in obj.items():
            if k=='brut': continue
            c += count_values(v)
    elif isinstance(obj, list): c = sum(count_values(v) for v in obj)
    elif isinstance(obj,(int,float)) and not isinstance(obj,bool): c = 1
    elif isinstance(obj,str) and obj: c = 1
    return c
print('✅ Normalisation OK')

In [ ]:
# CONSTRUCTION PAGES JSON
def build_blank_page(page, src):
    n = page['index']+1
    return {'page_id':'p'+str(n).zfill(3),'pdf_page_index':page['index'],'numero_page_scannee':n,'numero_page_imprimee':None,'fichier_source':src,
      'classification':{'type_page':'BLANCHE','types':[],'annexe_numeros':[],'sous_type_page':'PAGE_VIERGE','titre_page':None,'page_annexe':False,'page_utile':False,'page_blanche':True},
      'statut_extraction':{'statut':'BLANCHE','nb_tableaux':0,'nb_champs_extraits':0,'commentaire':'Page blanche, non envoyee au VLM.'},'entete_page':{},'donnees':{},'tokens_in':0,'tokens_out':0,'temps_s':0.0}
def build_page_object(page, types, data, rep, src):
    n = page['index']+1; data = data if isinstance(data, dict) else {}
    donnees = {'brut': data}
    for t in types: donnees[t] = normalise_table(t, data.get(t))
    return {'page_id':'p'+str(n).zfill(3),'pdf_page_index':page['index'],'numero_page_scannee':n,
      'numero_page_imprimee': data.get('numero_page_imprimee') if isinstance(data.get('numero_page_imprimee'), int) else None,'fichier_source':src,
      'classification':{'type_page': types[0] if types else 'AUTRE','types':types,'annexe_numeros': page.get('annexe_nums') or [],
        'sous_type_page':' '.join(types) if types else 'AUTRE','titre_page': norm_str(data.get('titre_page')),
        'page_annexe': any(t.startswith('A') or t in ('ST','REM','DIST') for t in types),'page_utile': bool(types),'page_blanche':False},
      'statut_extraction':{'statut':'OK' if data else 'ECHEC_EXTRACTION','nb_tableaux':len(types),'nb_champs_extraits':count_values(donnees),'commentaire':None},
      'entete_page': normalise_entete(data),'donnees':donnees,
      'tokens_in': rep.get('tokens_in') if rep else 0,'tokens_out': rep.get('tokens_out') if rep else 0,'temps_s': rep.get('elapsed') if rep else 0.0}
def merge_core(base, new):
    if new is None: return base
    if base is None: return copy.deepcopy(new)
    for key, cols in new.items():
        if key not in base: base[key] = copy.deepcopy(cols)
        elif isinstance(cols, dict):
            for col, val in cols.items():
                if base[key].get(col) is None and val is not None: base[key][col] = val
    return base
def build_synthese(pobjs):
    syn = {'actif':None,'passif':None,'tcr':None,'decl':None,'annexes':{},'annexes_pages':[]}
    for p in pobjs:
        types = p['classification'].get('types') or []; d = p.get('donnees', {})
        if 'ACTIF' in types and syn['actif'] is None: syn['actif'] = d.get('ACTIF')
        if 'PASSIF' in types and syn['passif'] is None: syn['passif'] = d.get('PASSIF')
        if 'TCR' in types: syn['tcr'] = merge_core(syn['tcr'], d.get('TCR'))
        if 'DECL' in types and syn['decl'] is None: syn['decl'] = d.get('DECL')
        for t in types:
            if t in ('ACTIF','PASSIF','TCR','DECL','AUTRE'): continue
            if syn['annexes'].get(t) is None and d.get(t): syn['annexes'][t] = d.get(t)
        if any(t not in ('ACTIF','PASSIF','TCR','DECL','AUTRE') for t in types):
            syn['annexes_pages'].append({'page_id':p['page_id'],'numero_page_scannee':p['numero_page_scannee'],'types':types,'annexe_numeros':p['classification'].get('annexe_numeros')})
    return syn
def build_document(pdf_path, pages, pobjs, ti, to, elapsed):
    return {'schema_version':'16.0','type_document':'liasse_fiscale_algerienne_serie_g',
      'document':{'document_id':'doc_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+pdf_path.stem,'fichier_source':pdf_path.name,
        'nb_pages_pdf':len(pages),'nb_pages_scannes':len(pages),
        'nb_pages_blanches':sum(1 for p in pobjs if p['classification']['type_page']=='BLANCHE'),
        'nb_pages_utiles':sum(1 for p in pobjs if p['classification']['type_page']!='BLANCHE'),
        'nb_pages_annexes':sum(1 for p in pobjs if p['classification'].get('page_annexe')),
        'date_extraction':datetime.now().strftime('%Y-%m-%d %H:%M:%S'),'duree_extraction_s':round(elapsed,2),
        'modele_extraction':MODEL_PATH.split('/')[-1],'prompt_version':'v16','referentiel_version':'liasse_serie_g_v16',
        'tokens_in':ti,'tokens_out':to,'tokens_total':ti+to},
      'pages':pobjs,'synthese':build_synthese(pobjs)}
print('✅ Construction pages OK')

In [ ]:
# IDENTITE DOSSIER (NIF15, millesimes, concordance pages)
def nif_15(v):
    if v is None: return None, 'ABSENT'
    c = re.sub(r'\D','',str(v))
    if not c: return None, 'ABSENT'
    if len(c)==15: return c, 'CONFORME'
    if len(c)<15: return c.zfill(15), 'COMPLETE'
    return c[:15], 'TRONQUE'
def cle_nom(v):
    if not v: return None
    s = ''.join(c for c in unicodedata.normalize('NFD', str(v)) if unicodedata.category(c)!='Mn').upper()
    s = re.sub(r'\b(SARL|EURL|SPA|SNC|ETS|STE|SOCIETE|GROUPE)\b',' ', s)
    return ' '.join(re.sub(r'[^A-Z0-9]+',' ', s).split()) or None
def annee_exercice(ent):
    for k in ('exercice','exercice_au','exercice_du'):
        v = ent.get(k)
        if v:
            m = re.findall(r'(19|20)\d{2}', str(v))
            if m: return int(m[-1])
    return None
def date_cloture(ent):
    for k in ('exercice','exercice_au'):
        v = ent.get(k)
        if v and re.search(r'\d{2}/\d{2}/\d{4}', str(v)): return re.search(r'\d{2}/\d{2}/\d{4}', str(v)).group()
    return norm_str(ent.get('exercice'))
def _majoritaire(vals):
    vals = [v for v in vals if v not in (None,'')]
    if not vals: return None,0,0
    val,n = Counter(vals).most_common(1)[0]; return val,n,len(vals)
def construire_identite(doc):
    par_page, alertes = [], []
    for page in doc.get('pages', []):
        if page['classification'].get('page_blanche'): continue
        ent = page.get('entete_page') or {}
        n15, st = nif_15(ent.get('nif'))
        par_page.append({'page_id':page['page_id'],'numero_page_scannee':page['numero_page_scannee'],'types':page['classification'].get('types') or [],
          'entreprise':norm_str(ent.get('entreprise')),'entreprise_cle':cle_nom(ent.get('entreprise')),'nif_brut':norm_str(ent.get('nif')),
          'nif_15':n15,'nif_statut':st,'exercice_clos':date_cloture(ent),'annee_n':annee_exercice(ent)})
    nif_ret,nn,tn = _majoritaire([p['nif_15'] for p in par_page])
    nom_ret,nno,tno = _majoritaire([p['entreprise_cle'] for p in par_page])
    an_ret,nan,tan = _majoritaire([p['annee_n'] for p in par_page])
    libelle = None
    for p in par_page:
        if p['entreprise_cle']==nom_ret and p['entreprise']:
            if libelle is None or len(p['entreprise'])>len(libelle): libelle = p['entreprise']
    cloture,_,_ = _majoritaire([p['exercice_clos'] for p in par_page])
    def div(champ, retenu): return [{'page_id':p['page_id'],'numero_page_scannee':p['numero_page_scannee'],'valeur':p[champ]} for p in par_page if p[champ] not in (None,'') and p[champ]!=retenu]
    for champ, retenu, lib, grav in [('nif_15',nif_ret,'NIF','critique'),('entreprise_cle',nom_ret,'raison sociale','critique'),('annee_n',an_ret,'exercice clos','majeur')]:
        d = div(champ, retenu)
        if d: alertes.append({'code':'IDENTITE_DIVERGENTE','champ':lib,'gravite':grav,'valeur_retenue':retenu,'pages_divergentes':d,'message':str(len(d))+' page(s) divergente(s) sur '+lib+'.'})
    couverture = {}
    for champ, lib in [('nif_15','NIF'),('entreprise','raison sociale'),('annee_n','exercice clos')]:
        absentes = [p['page_id'] for p in par_page if not p[champ]]
        couverture[champ] = {'pages_renseignees':len(par_page)-len(absentes),'pages_totales':len(par_page),'pages_absentes':absentes}
        if absentes: alertes.append({'code':'ENTETE_INCOMPLETE','champ':lib,'gravite':'mineur','pages_sans_valeur':absentes,'message':str(len(absentes))+' page(s) sans '+lib+'.'})
    orph = [p['page_id'] for p in par_page if not p['nif_15'] and not p['entreprise'] and not p['annee_n']]
    if orph: alertes.append({'code':'PAGE_ORPHELINE','champ':'en-tete','gravite':'majeur','pages_sans_valeur':orph,'message':str(len(orph))+' page(s) sans identite.'})
    doc['identite'] = {'entreprise':libelle,'entreprise_cle':nom_ret,'nif_15':nif_ret,'exercice_clos':cloture,'annee_n':an_ret,
      'annee_n1':(an_ret-1) if isinstance(an_ret,int) else None,'libelle_n':('N : '+str(an_ret)) if an_ret else 'N','libelle_n1':('N-1 : '+str(an_ret-1)) if an_ret else 'N-1',
      'concordance':{'nif':{'pages_concordantes':nn,'pages_renseignees':tn},'entreprise':{'pages_concordantes':nno,'pages_renseignees':tno},'exercice':{'pages_concordantes':nan,'pages_renseignees':tan}},
      'couverture_entete':couverture,'dossier_homogene':not any(a['gravite']=='critique' for a in alertes),'pages':par_page,'alertes':alertes}
    return doc['identite']
print('✅ Identité OK')

In [ ]:
# REGLES : formules internes (dont sous-totaux ACTIF) + contrôles croisés
COLS_ACTIF = ['montant_brut','amortissements_provisions_pertes','net_n','net_n1']; COLS_PASSIF = ['n','n1']
FORMULES = [
 ('ACTIF','immobilisations_corporelles',COLS_ACTIF,[('+','terrains'),('+','batiments'),('+','autres_immobilisations_corporelles'),('+','immobilisations_en_concession')]),
 ('ACTIF','immobilisations_financieres',COLS_ACTIF,[('+','titres_mis_en_equivalence'),('+','autres_participations_creances'),('+','autres_titres_immobilises'),('+','prets_actifs_financiers_non_courants'),('+','impots_differes_actif')]),
 ('ACTIF','creances_et_emplois_assimiles',COLS_ACTIF,[('+','clients'),('+','autres_debiteurs'),('+','impots_assimiles_actif'),('+','autres_creances_assimiles')]),
 ('ACTIF','disponibilites_et_assimiles',COLS_ACTIF,[('+','placements_financiers_courants'),('+','tresorerie_actif')]),
 ('ACTIF','total_actif_non_courant',COLS_ACTIF,[('+','ecarts_acquisition_goodwill'),('+','immobilisations_incorporelles'),('+','immobilisations_corporelles'),('+','immobilisations_en_concession'),('+','immobilisations_en_cours'),('+','immobilisations_financieres')]),
 ('ACTIF','total_actif_courant',COLS_ACTIF,[('+','stocks_encours'),('+','creances_et_emplois_assimiles'),('+','disponibilites_et_assimiles')]),
 ('ACTIF','total_general_actif',COLS_ACTIF,[('+','total_actif_non_courant'),('+','total_actif_courant')]),
 ('PASSIF','total_capitaux_propres',COLS_PASSIF,[('+','capital_emis'),('-','capital_non_appele'),('+','primes_reserves'),('+','ecart_reevaluation'),('+','ecart_equivalence'),('+','resultat_net_passif'),('+','report_a_nouveau'),('+','part_societe_consolidante'),('+','part_minoritaires')]),
 ('PASSIF','total_passifs_non_courants',COLS_PASSIF,[('+','emprunts_dettes_financieres'),('+','impots_differes_provisionnes'),('+','autres_dettes_non_courantes'),('+','provisions_produits_avance')]),
 ('PASSIF','total_passifs_courants',COLS_PASSIF,[('+','fournisseurs_rattaches'),('+','impots_passif'),('+','autres_dettes'),('+','tresorerie_passif')]),
 ('PASSIF','total_general_passif',COLS_PASSIF,[('+','total_capitaux_propres'),('+','total_passifs_non_courants'),('+','total_passifs_courants')]),
 ('TCR','chiffre_affaires_net','NET',[('+','ventes_marchandises'),('+','produits_fabriques'),('+','prestations_services'),('+','ventes_travaux'),('+','produits_annexes'),('+','rabais_remises_ristournes_accordes')]),
 ('TCR','production_exercice','NET',[('+','chiffre_affaires_net'),('+','production_stockee_destockee'),('+','production_immobilisee'),('+','subvention_exploitation')]),
 ('TCR','consommations_exercice','NET',[('+','achats_marchandises_vendues'),('+','matieres_premieres'),('+','autres_approvisionnements'),('+','variation_stocks'),('+','achats_etudes_prestations'),('+','autres_consommations'),('-','rabais_remises_obtenus_achats'),('+','sous_traitance_generale'),('+','locations'),('+','entretien_reparations'),('+','primes_assurances'),('+','personnel_exterieur'),('+','remuneration_intermediaires'),('+','publicite'),('+','deplacements_missions'),('+','autres_services'),('-','rabais_remises_obtenus_services')]),
 ('TCR','valeur_ajoutee_exploitation','NET',[('+','production_exercice'),('+','consommations_exercice')]),
 ('TCR','excedent_brut_exploitation','NET',[('+','valeur_ajoutee_exploitation'),('+','charges_personnel'),('+','impots_taxes_assimiles')]),
 ('TCR','resultat_operationnel','NET',[('+','excedent_brut_exploitation'),('+','autres_produits_operationnels'),('+','autres_charges_operationnelles'),('+','dotations_amortissements'),('+','provisions'),('+','pertes_valeur'),('+','reprises_pertes_valeur_provisions')]),
 ('TCR','resultat_financier','NET',[('+','produits_financiers'),('+','charges_financieres')]),
 ('TCR','resultat_ordinaire','NET',[('+','resultat_operationnel'),('+','resultat_financier')]),
 ('TCR','resultat_extraordinaire','NET',[('+','elements_extraordinaires_produits'),('+','elements_extraordinaires_charges')]),
 ('TCR','resultat_net_exercice','NET',[('+','resultat_ordinaire'),('+','resultat_extraordinaire'),('+','impots_exigibles_resultats'),('+','impots_differes_resultats')]),
 ('A1','total',['solde_debut','debit','credit','solde_fin'],[('+','stocks_marchandises'),('+','matieres_fournitures'),('+','autres_approvisionnements'),('+','encours_production_biens'),('+','encours_production_services'),('+','stocks_produits'),('+','stocks_provenant_immobilisations'),('+','stocks_exterieur')]),
 ('A3','total_autres_services',['montant'],[('+','charges_locatives'),('+','etudes_recherches'),('+','documentation_divers'),('+','transports_biens'),('+','frais_postaux'),('+','services_bancaires'),('+','cotisations_divers')]),
 ('A3','total_charges_personnel',['montant'],[('+','remunerations_personnel'),('+','remuneration_exploitant'),('+','cotisations_sociales'),('+','charges_sociales_exploitant'),('+','autres_charges_sociales'),('+','autres_charges_personnel')]),
 ('A3','total_impots',['montant'],[('+','impots_sur_remunerations'),('+','impots_non_recuperables'),('+','autres_impots_taxes')]),
 ('A3','total_general',['montant'],[('+','total_autres_services'),('+','total_charges_personnel'),('+','total_impots')]),
 ('A4','total_charges',['montant'],[('+','redevances_concessions_charges'),('+','moins_values_sorties_actifs'),('+','jetons_presence_charges'),('+','pertes_creances_irrecouvrables'),('+','quote_part_operations_commun_charges'),('+','amendes_penalites_dons'),('+','charges_exceptionnelles_gestion'),('+','autres_charges_gestion')]),
 ('A4','total_produits',['montant'],[('+','redevances_concessions_produits'),('+','plus_values_sorties_actifs'),('+','jetons_presence_produits'),('+','quotes_parts_subventions_virees'),('+','quote_part_operations_commun_produits'),('+','rentrees_creances_amorties'),('+','produits_exceptionnels_gestion'),('+','autres_produits_gestion')]),
 ('A5','total',['dotations_cumulees_debut','dotations_exercice','diminutions_elements_sortis','dotations_cumulees_fin','dotations_fiscales_exercice','ecarts'],[('+','goodwill'),('+','immobilisations_incorporelles'),('+','immobilisations_corporelles'),('+','participations'),('+','autres_actifs_financiers_non_courants')]),
 ('A8','total',['provisions_cumulees_debut','dotations_exercice','reprises_exercice','provisions_cumulees_fin'],[('+','pertes_valeur_stocks'),('+','pertes_valeur_creances'),('+','pertes_valeur_actions'),('+','provisions_pensions'),('+','provisions_litiges'),('+','autres_provisions_personnel'),('+','provisions_impots'),('+','autres_provisions')]),
 ('A9','total_reintegrations',['montant'],[('+','charges_immeubles_non_affectes'),('+','quote_part_cadeaux_publicitaires'),('+','quote_part_sponsoring'),('+','frais_reception'),('+','cotisations_dons'),('+','impots_taxes_non_deductibles'),('+','provisions_non_deductibles'),('+','amortissements_non_deductibles'),('+','quote_part_frais_rd'),('+','amortissements_credit_bail_preneur'),('+','loyers_hors_produits_financiers_bailleur'),('+','ibs_impot_exigible'),('+','ibs_impot_differe'),('+','pertes_valeur_non_deductibles'),('+','amendes_penalites'),('+','autres_reintegrations')]),
 ('A9','total_deductions',['montant'],[('+','plus_values_cession_actif_immobilise'),('+','produits_plus_values_actions_bourse'),('+','revenus_distribution_benefices'),('+','amortissements_credit_bail_bailleur'),('+','loyers_hors_charges_financieres_preneur'),('+','complement_amortissements'),('+','autres_deductions')]),
 ('A10','origine_total',['montant'],[('+','origine_report_a_nouveau_n1'),('+','origine_resultat_n1'),('+','origine_prelevements_reserves')]),
 ('A10','affectation_total',['montant'],[('+','affectation_reserves'),('+','affectation_augmentation_capital'),('+','affectation_dividendes'),('+','affectation_report_a_nouveau')])]
FORMULES_LIGNE = [
 ('ACTIF','net_n',[('+','montant_brut'),('-','amortissements_provisions_pertes')]),
 ('A1','solde_fin',[('+','solde_debut'),('+','debit'),('-','credit')]),
 ('A5','dotations_cumulees_fin',[('+','dotations_cumulees_debut'),('+','dotations_exercice'),('-','diminutions_elements_sortis')]),
 ('A5','ecarts',[('+','dotations_exercice'),('-','dotations_fiscales_exercice')]),
 ('A7','valeur_nette_comptable',[('+','montant_net_actif'),('-','amortissements_pratiques')]),
 ('A8','provisions_cumulees_fin',[('+','provisions_cumulees_debut'),('+','dotations_exercice'),('-','reprises_exercice')]),
 ('A82','valeur_nette_comptable',[('+','valeur_nominale_debut'),('-','perte_valeur_constituee')])]
CONTROLES = [
 ('Equilibre du bilan (N)','critique',[('+','ACTIF','total_general_actif','net_n')],[('+','PASSIF','total_general_passif','n')]),
 ('Equilibre du bilan (N-1)','critique',[('+','ACTIF','total_general_actif','net_n1')],[('+','PASSIF','total_general_passif','n1')]),
 ('Resultat net : TCR = bilan passif (N)','critique',[('+','TCR','resultat_net_exercice','NET')],[('+','PASSIF','resultat_net_passif','n')]),
 ('Resultat net : TCR = bilan passif (N-1)','critique',[('+','TCR','resultat_net_exercice','NET_N1')],[('+','PASSIF','resultat_net_passif','n1')]),
 ('Resultat net : TCR = ligne I tableau 9','critique',[('+','TCR','resultat_net_exercice','NET')],[('+','A9','resultat_net_benefice','montant'),('-','A9','resultat_net_perte','montant')]),
 ('Stocks : total A1 (fin) = stocks bilan (N)','majeur',[('+','A1','total','solde_fin')],[('+','ACTIF','stocks_encours','net_n')]),
 ('Stocks : total A1 (debut) = stocks bilan (N-1)','majeur',[('+','A1','total','solde_debut')],[('+','ACTIF','stocks_encours','net_n1')]),
 ('Amortissements : cumul fin A5 = colonne amort. bilan','majeur',[('+','A5','total','dotations_cumulees_fin')],[('+','ACTIF','total_actif_non_courant','amortissements_provisions_pertes')]),
 ('Amortissements : dotations A5 = dotations TCR','majeur',[('+','A5','total','dotations_exercice')],[('+','TCR','dotations_amortissements','n_debit')]),
 ('Charges A3 (2) = charges personnel TCR','majeur',[('+','A3','total_charges_personnel','montant')],[('+','TCR','charges_personnel','n_debit')]),
 ('Charges A3 (3) = impots et taxes TCR','majeur',[('+','A3','total_impots','montant')],[('+','TCR','impots_taxes_assimiles','n_debit')]),
 ('A4 charges = autres charges TCR','majeur',[('+','A4','total_charges','montant')],[('+','TCR','autres_charges_operationnelles','n_debit')]),
 ('A4 produits = autres produits TCR','majeur',[('+','A4','total_produits','montant')],[('+','TCR','autres_produits_operationnels','n_credit')]),
 ('Provisions : dotations A8 = provisions+pertes TCR','majeur',[('+','A8','total','dotations_exercice')],[('+','TCR','provisions','n_debit'),('+','TCR','pertes_valeur','n_debit')]),
 ('Provisions : reprises A8 = reprises TCR','majeur',[('+','A8','total','reprises_exercice')],[('+','TCR','reprises_pertes_valeur_provisions','n_credit')]),
 ('Fiscal : resultat = I + reint - ded - deficits','critique',[('+','A9','resultat_fiscal_benefice','montant'),('-','A9','resultat_fiscal_deficit','montant')],[('+','A9','resultat_net_benefice','montant'),('-','A9','resultat_net_perte','montant'),('+','A9','total_reintegrations','montant'),('-','A9','total_deductions','montant'),('-','A9','total_deficits_a_deduire','montant')]),
 ('Fiscal : resultat fiscal DECL = tableau 9','majeur',[('+','DECL','resultat_fiscal','valeur')],[('+','A9','resultat_fiscal_benefice','montant'),('-','A9','resultat_fiscal_deficit','montant')]),
 ('Fiscal : CA global DECL = CA net TCR','majeur',[('+','DECL','chiffre_affaires_global_ht','valeur')],[('+','TCR','chiffre_affaires_net','NET')]),
 ('Affectation : origine = affectation','critique',[('+','A10','origine_total','montant')],[('+','A10','affectation_total','montant')]),
 ('Affectation : resultat N-1 = RN bilan (N-1)','majeur',[('+','A10','origine_resultat_n1','montant')],[('+','PASSIF','resultat_net_passif','n1')]),
 ('Tableau 12 = remuneration intermediaires TCR','majeur',[('+','A12',' TOTAL ','montant_percu')],[('+','TCR','remuneration_intermediaires','n_debit')]),
 ('Distributions : global = societe + etablissement','majeur',[('+','DIST','montant_global_brut','montant')],[('+','DIST','paye_par_societe','montant'),('+','DIST','paye_par_etablissement','montant')])]
print('✅ Règles OK —', len(FORMULES), 'formules,', len(FORMULES_LIGNE), 'formules ligne,', len(CONTROLES), 'contrôles')

In [ ]:
# MOTEUR DE COHERENCE : recalcul, comparaison, enrichment valeur_certaine/ecart
TOLERANCE_DA = 1.0
def lire_bloc(bloc):
    if isinstance(bloc, list):
        return [('ligne_'+str(i).zfill(3), {k:v for k,v in row.items() if k!='libelle_imprime'}, row.get('libelle_imprime')) for i,row in enumerate(bloc) if isinstance(row,dict)]
    if isinstance(bloc, dict):
        if isinstance(bloc.get('lignes'), list):
            return [(lg.get('row_code') or ('ligne_'+str(i).zfill(3)), lg.get('valeurs') or {}, lg.get('libelle_imprime')) for i,lg in enumerate(bloc['lignes']) if isinstance(lg,dict)]
        return [(rc, vals, None) for rc, vals in bloc.items() if isinstance(vals, dict)]
    return []
def iter_blocs(doc):
    for page in doc.get('pages', []):
        for tab, bloc in (page.get('donnees') or {}).items():
            if tab in ('brut','AUTRE') or bloc in (None,{},[]): continue
            yield page, tab, bloc
def _index_postes(doc):
    idx, dyn = {}, {}
    for _p, tab, bloc in iter_blocs(doc):
        for cle, vals, _lib in lire_bloc(bloc):
            rens = {k:v for k,v in (vals or {}).items() if v is not None}
            if rens: idx.setdefault((tab,cle),{}).update(rens)
            if cle != 'total':
                for col, v in rens.items():
                    if isinstance(v,(int,float)) and not isinstance(v,bool): dyn[(tab,col)] = dyn.get((tab,col),0.0)+float(v)
    for (tab,col), s in dyn.items(): idx.setdefault((tab,' TOTAL '),{})[col] = s
    return idx
def _valeur(idx, tab, rc, col):
    vals = idx.get((tab,rc))
    if vals is None: return None
    if col in ('NET','NET_N1'):
        suf = ('n_credit','n_debit') if col=='NET' else ('n1_credit','n1_debit')
        c,d = vals.get(suf[0]), vals.get(suf[1])
        if c is None and d is None: return None
        return float(c or 0) - float(d or 0)
    v = vals.get(col)
    return float(v) if isinstance(v,(int,float)) and not isinstance(v,bool) else None
def _somme(idx, termes):
    tot, vus = 0.0, 0
    for signe, tab, rc, col in termes:
        v = _valeur(idx, tab, rc, col)
        if v is not None: tot += v if signe=='+' else -v; vus += 1
    return tot if vus else None
def _statut(e, tol):
    if abs(e) <= 1e-9: return 'coherent'
    if abs(e) <= tol: return 'coherent_arrondi'
    return 'ecart_significatif'
def verifier_coherence(doc, tol=TOLERANCE_DA):
    idx = _index_postes(doc); rf, rc_ = [], []
    for tab, cible, cols, comps in FORMULES:
        for col in (['NET','NET_N1'] if cols=='NET' else cols):
            dec = _valeur(idx, tab, cible, col); rec = _somme(idx, [(s,tab,r,col) for s,r in comps])
            if dec is None or rec is None: continue
            e = dec - rec
            rf.append({'type':'agregat','tableau':tab,'poste':cible,'colonne':col,'valeur_extraite':dec,'valeur_recalculee':rec,'ecart':round(e,2),'statut':_statut(e,tol)})
    for tab, col_cible, comps in FORMULES_LIGNE:
        for (t, rc) in list(idx.keys()):
            if t != tab or rc==' TOTAL ': continue
            dec = _valeur(idx, tab, rc, col_cible); rec = _somme(idx, [(s,tab,rc,c) for s,c in comps])
            if dec is None or rec is None: continue
            e = dec - rec
            rf.append({'type':'ligne','tableau':tab,'poste':rc,'colonne':col_cible,'valeur_extraite':dec,'valeur_recalculee':rec,'ecart':round(e,2),'statut':_statut(e,tol)})
    for lib, grav, g, d in CONTROLES:
        a, b = _somme(idx, g), _somme(idx, d)
        if a is None or b is None:
            rc_.append({'controle':lib,'gravite':grav,'statut':'non_verifiable','valeur_a':a,'valeur_b':b,'ecart':None,'commentaire':'Un des deux membres est absent.'}); continue
        e = a - b
        rc_.append({'controle':lib,'gravite':grav,'valeur_a':round(a,2),'valeur_b':round(b,2),'ecart':round(e,2),'statut':_statut(e,tol)})
    ec_f = [f for f in rf if f['statut']=='ecart_significatif']; ec_c = [c for c in rc_ if c['statut']=='ecart_significatif']
    crit = [c for c in ec_c if c['gravite']=='critique']
    return {'tolerance_da':tol,'formules':rf,'controles_croises':rc_,'synthese':{'formules_verifiees':len(rf),'formules_en_ecart':len(ec_f),'controles_verifies':len([c for c in rc_ if c['statut']!='non_verifiable']),'controles_non_verifiables':len([c for c in rc_ if c['statut']=='non_verifiable']),'controles_en_ecart':len(ec_c),'ecarts_critiques':len(crit),'arbitrage_requis':bool(ec_f or ec_c),'liasse_exploitable':not crit}}
def enrichir_lignes(doc, rapport):
    index = {(f['tableau'],f['poste'],f['colonne']): f for f in rapport['formules']}
    for page, tab, bloc in iter_blocs(doc):
        cible = page.setdefault('controles_donnees', {}).setdefault(tab, {})
        for cle, vals, _lib in lire_bloc(bloc):
            ctrl = {}
            for col, val in (vals or {}).items():
                if val is None or not isinstance(val,(int,float)) or isinstance(val,bool): continue
                f = index.get((tab,cle,col))
                ctrl[col] = ({'valeur_certaine': f['statut']!='ecart_significatif','valeur_extraite':f['valeur_extraite'],'valeur_recalculee':f['valeur_recalculee'],'ecart':f['ecart'],'statut':f['statut']} if f else {'valeur_certaine':None,'statut':'non_verifiable'})
            if ctrl: cible[cle] = ctrl
        if not cible: page['controles_donnees'].pop(tab, None)
    return doc
def appliquer_controles(doc):
    ident = construire_identite(doc); rap = verifier_coherence(doc); enrichir_lignes(doc, rap)
    doc['controles'] = rap; doc['schema_version'] = '16.0'
    s = rap['synthese']; doc['controles']['synthese']['identite_homogene'] = ident['dossier_homogene']
    doc['controles']['synthese']['arbitrage_requis'] = (s['arbitrage_requis'] or not ident['dossier_homogene'])
    return doc
print('✅ Moteur de cohérence OK')

In [ ]:
# MAPPING EXCEL : sous-totaux ACTIF en formules (True = jamais écrire)
MAP_FEUILLES = {'ACTIF':'1. Bilan Actif','PASSIF':'2. Bilan Passif','TCR':'3. TCR','A1':'4. Stocks','A2':'4. Stocks','A3':'5. Charges & produits','A4':'5. Charges & produits','A5':'6. Amort. & Immo.','A6':'6. Amort. & Immo.','A7':'7. Cessions & Provisions','A8':'7. Cessions & Provisions','A81':'8. Pertes de valeurs','A82':'8. Pertes de valeurs','A9':'9. Résultat fiscal','A10':'10. Affectation & Particip.','A11':'10. Affectation & Particip.','A12':'11. Commissions & TAP','A13':'11. Commissions & TAP'}
MAP_COLONNES = {'ACTIF':{'montant_brut':'B','amortissements_provisions_pertes':'C','net_n':'D','net_n1':'E'},'PASSIF':{'n':'B','n1':'C'},'TCR':{'n_debit':'B','n_credit':'C','n1_debit':'D','n1_credit':'E'},'A1':{'solde_debut':'B','debit':'C','credit':'D','solde_fin':'E'},'A2':{'debit':'B','credit':'C','solde_debiteur':'D','solde_crediteur':'E'},'A3':{'montant':'B'},'A4':{'montant':'B'},'A5':{'dotations_cumulees_debut':'B','dotations_exercice':'C','diminutions_elements_sortis':'D','dotations_cumulees_fin':'E','dotations_fiscales_exercice':'F','ecarts':'G'},'A6':{'montants_bruts':'B','tva_deduite':'C','montant_net_a_amortir':'D'},'A7':{'date_acquisition':'B','montant_net_actif':'C','amortissements_pratiques':'D','valeur_nette_comptable':'E','prix_cession':'F','plus_value':'G','moins_value':'H'},'A8':{'provisions_cumulees_debut':'B','dotations_exercice':'C','reprises_exercice':'D','provisions_cumulees_fin':'E'},'A81':{'valeur_creance':'B','perte_valeur_constituee':'C'},'A82':{'valeur_nominale_debut':'B','perte_valeur_constituee':'C','valeur_nette_comptable':'D'},'A9':{'montant':'B'},'A10':{'montant':'B'},'A11':{'capitaux_propres':'B','dont_capital':'C','quote_part_capital_pct':'D','resultat_dernier_exercice':'E','prets_avances':'F','dividendes_encaisses':'G','valeur_comptable_titres':'H'},'A12':{'nif':'B','adresse':'C','montant_percu':'D'},'A13':{'ca_imposable':'B','ca_exonere':'C','tap_acquittee':'D'}}
MAP_LIGNES = {
 'ACTIF': {'ecarts_acquisition_goodwill':(11,False),'immobilisations_incorporelles':(12,False),'immobilisations_corporelles':(13,True),'terrains':(14,False),'batiments':(15,False),'autres_immobilisations_corporelles':(16,False),'immobilisations_en_concession':(17,False),'immobilisations_en_cours':(18,False),'immobilisations_financieres':(19,True),'titres_mis_en_equivalence':(20,False),'autres_participations_creances':(21,False),'autres_titres_immobilises':(22,False),'prets_actifs_financiers_non_courants':(23,False),'impots_differes_actif':(24,False),'total_actif_non_courant':(25,True),'stocks_encours':(27,False),'creances_et_emplois_assimiles':(28,True),'clients':(29,False),'autres_debiteurs':(30,False),'impots_assimiles_actif':(31,False),'autres_creances_assimiles':(32,False),'disponibilites_et_assimiles':(33,True),'placements_financiers_courants':(34,False),'tresorerie_actif':(35,False),'total_actif_courant':(36,True),'total_general_actif':(37,True)},
 'PASSIF': {'capital_emis':(10,False),'capital_non_appele':(11,False),'primes_reserves':(12,False),'ecart_reevaluation':(13,False),'ecart_equivalence':(14,False),'resultat_net_passif':(15,False),'report_a_nouveau':(16,False),'part_societe_consolidante':(17,False),'part_minoritaires':(18,False),'total_capitaux_propres':(19,True),'emprunts_dettes_financieres':(21,False),'impots_differes_provisionnes':(22,False),'autres_dettes_non_courantes':(23,False),'provisions_produits_avance':(24,False),'total_passifs_non_courants':(25,True),'fournisseurs_rattaches':(27,False),'impots_passif':(28,False),'autres_dettes':(29,False),'tresorerie_passif':(30,False),'total_passifs_courants':(31,True),'total_general_passif':(32,True)},
 'TCR': {'ventes_marchandises':(10,False),'produits_fabriques':(12,False),'prestations_services':(13,False),'ventes_travaux':(14,False),'produits_annexes':(15,False),'rabais_remises_ristournes_accordes':(16,False),'chiffre_affaires_net':(17,True),'production_stockee_destockee':(18,False),'production_immobilisee':(19,False),'subvention_exploitation':(20,False),'production_exercice':(21,True),'achats_marchandises_vendues':(22,False),'matieres_premieres':(23,False),'autres_approvisionnements':(24,False),'variation_stocks':(25,False),'achats_etudes_prestations':(26,False),'autres_consommations':(27,False),'rabais_remises_obtenus_achats':(28,False),'sous_traitance_generale':(30,False),'locations':(31,False),'entretien_reparations':(32,False),'primes_assurances':(33,False),'personnel_exterieur':(34,False),'remuneration_intermediaires':(35,False),'publicite':(36,False),'deplacements_missions':(37,False),'autres_services':(38,False),'rabais_remises_obtenus_services':(39,False),'consommations_exercice':(40,True),'valeur_ajoutee_exploitation':(41,True),'charges_personnel':(42,False),'impots_taxes_assimiles':(43,False),'excedent_brut_exploitation':(44,True),'autres_produits_operationnels':(45,False),'autres_charges_operationnelles':(46,False),'dotations_amortissements':(47,False),'provisions':(48,False),'pertes_valeur':(49,False),'reprises_pertes_valeur_provisions':(50,False),'resultat_operationnel':(51,True),'produits_financiers':(52,False),'charges_financieres':(53,False),'resultat_financier':(54,True),'resultat_ordinaire':(55,True),'elements_extraordinaires_produits':(56,False),'elements_extraordinaires_charges':(57,False),'resultat_extraordinaire':(58,True),'impots_exigibles_resultats':(59,False),'impots_differes_resultats':(60,False),'resultat_net_exercice':(61,True)},
 'A1': {'stocks_marchandises':(11,False),'matieres_fournitures':(12,False),'autres_approvisionnements':(13,False),'encours_production_biens':(14,False),'encours_production_services':(15,False),'stocks_produits':(16,False),'stocks_provenant_immobilisations':(17,False),'stocks_exterieur':(18,False),'total':(19,True)},
 'A3': {'charges_locatives':(11,False),'etudes_recherches':(12,False),'documentation_divers':(13,False),'transports_biens':(14,False),'frais_postaux':(15,False),'services_bancaires':(16,False),'cotisations_divers':(17,False),'total_autres_services':(18,True),'remunerations_personnel':(20,False),'remuneration_exploitant':(21,False),'cotisations_sociales':(22,False),'charges_sociales_exploitant':(23,False),'autres_charges_sociales':(24,False),'autres_charges_personnel':(25,False),'total_charges_personnel':(26,True),'impots_sur_remunerations':(28,False),'impots_non_recuperables':(29,False),'autres_impots_taxes':(30,False),'total_impots':(31,True),'total_general':(32,True)},
 'A4': {'redevances_concessions_charges':(37,False),'moins_values_sorties_actifs':(38,False),'jetons_presence_charges':(39,False),'pertes_creances_irrecouvrables':(40,False),'quote_part_operations_commun_charges':(41,False),'amendes_penalites_dons':(42,False),'charges_exceptionnelles_gestion':(43,False),'autres_charges_gestion':(44,False),'total_charges':(45,True),'redevances_concessions_produits':(50,False),'plus_values_sorties_actifs':(51,False),'jetons_presence_produits':(52,False),'quotes_parts_subventions_virees':(53,False),'quote_part_operations_commun_produits':(54,False),'rentrees_creances_amorties':(55,False),'produits_exceptionnels_gestion':(56,False),'autres_produits_gestion':(57,False),'total_produits':(58,True)},
 'A5': {'goodwill':(10,False),'immobilisations_incorporelles':(11,False),'immobilisations_corporelles':(12,False),'participations':(13,False),'autres_actifs_financiers_non_courants':(14,False),'total':(15,True)},
 'A6': {'goodwill':(20,False),'immobilisations_incorporelles':(21,False),'immobilisations_corporelles':(22,False),'participations':(23,False),'autres_actifs_financiers_non_courants':(24,False),'total':(25,True)},
 'A8': {'pertes_valeur_stocks':(26,False),'pertes_valeur_creances':(27,False),'pertes_valeur_actions':(28,False),'provisions_pensions':(29,False),'provisions_litiges':(30,False),'autres_provisions_personnel':(31,False),'provisions_impots':(32,False),'autres_provisions':(33,False),'total':(34,True)},
 'A9': {'resultat_net_benefice':(9,False),'resultat_net_perte':(10,False),'charges_immeubles_non_affectes':(12,False),'quote_part_cadeaux_publicitaires':(13,False),'quote_part_sponsoring':(14,False),'frais_reception':(15,False),'cotisations_dons':(16,False),'impots_taxes_non_deductibles':(17,False),'provisions_non_deductibles':(18,False),'amortissements_non_deductibles':(19,False),'quote_part_frais_rd':(20,False),'amortissements_credit_bail_preneur':(21,False),'loyers_hors_produits_financiers_bailleur':(22,False),'ibs_impot_exigible':(23,False),'ibs_impot_differe':(24,False),'pertes_valeur_non_deductibles':(25,False),'amendes_penalites':(26,False),'autres_reintegrations':(27,False),'total_reintegrations':(28,True),'plus_values_cession_actif_immobilise':(30,False),'produits_plus_values_actions_bourse':(31,False),'revenus_distribution_benefices':(32,False),'amortissements_credit_bail_bailleur':(33,False),'loyers_hors_charges_financieres_preneur':(34,False),'complement_amortissements':(35,False),'autres_deductions':(36,False),'total_deductions':(37,True),'total_deficits_a_deduire':(43,True),'resultat_fiscal_benefice':(44,True),'resultat_fiscal_deficit':(45,True)},
 'A10': {'origine_report_a_nouveau_n1':(11,False),'origine_resultat_n1':(12,False),'origine_prelevements_reserves':(13,False),'origine_total':(14,True),'affectation_reserves':(16,False),'affectation_augmentation_capital':(17,False),'affectation_dividendes':(18,False),'affectation_report_a_nouveau':(19,False),'affectation_total':(20,True)}}
MAP_DYNAMIQUES = {'A2':(25,32,None),'A7':(11,20,'A'),'A81':(10,23,'A'),'A82':(29,42,'A'),'A11':(26,34,'A'),'A12':(10,24,'A'),'A13':(30,41,'A')}
MAP_ENTETE = {'nif':'B4','entreprise':'B5','exercice':'B6'}
MAP_MILLESIMES = [('1. Bilan Actif','B8','N'),('1. Bilan Actif','E8','N1'),('2. Bilan Passif','B8','N'),('2. Bilan Passif','C8','N1'),('3. TCR','B8','N'),('3. TCR','D8','N1')]
print('✅ Mapping OK')

In [ ]:
# TRANSPOSITION : jamais écraser formules, arrondi ±1 jaune, écart significatif rouge, traçabilité PDF+page, verrouillage formules
_ROUGE = Border(*[Side(style='medium', color='FF0000')]*4); _ROUGE_FOND = PatternFill('solid', start_color='FFC7CE'); _ROUGE_TEXTE = Font(bold=True, color='9C0006')
_JAUNE_FOND = PatternFill('solid', start_color='FFF2CC')
_LIBRE = Protection(locked=False); _VERROU = Protection(locked=True)
def _est_formule(c): return isinstance(c.value, str) and c.value.startswith('=')
def _postes_du_document(doc):
    fixes, dyn = [], {}
    for _p, tab, bloc in iter_blocs(doc):
        for cle, vals, lib in lire_bloc(bloc):
            vals = vals or {}; est_libre = str(cle).startswith('ligne_')
            if not est_libre and tab in MAP_LIGNES and cle in MAP_LIGNES[tab]:
                for col, v in vals.items():
                    if v is not None: fixes.append((tab,cle,col,v))
            elif tab in MAP_DYNAMIQUES:
                if any(v is not None for v in vals.values()) or lib: dyn.setdefault(tab,[]).append({'libelle':lib,'valeurs':vals})
    return fixes, dyn
def _ecrire_valeurs(wb, doc):
    fixes, dyn = _postes_du_document(doc); ecrits, totaux, ignores = 0, [], []; par = {}
    ident = doc.get('identite') or {}
    ws1 = wb[MAP_FEUILLES['ACTIF']]
    if ident.get('nif_15'): ws1[MAP_ENTETE['nif']] = ident['nif_15']
    if ident.get('entreprise'): ws1[MAP_ENTETE['entreprise']] = ident['entreprise']
    if ident.get('exercice_clos'): ws1[MAP_ENTETE['exercice']] = ident['exercice_clos']
    for feuille, coord, quel in MAP_MILLESIMES:
        an = ident.get('annee_n') if quel=='N' else ident.get('annee_n1')
        if an: wb[feuille][coord] = ('N : '+str(an)) if quel=='N' else ('N-1 : '+str(an))
    for tab, rc, col, val in fixes:
        li = MAP_LIGNES.get(tab,{}).get(rc); colonne = MAP_COLONNES.get(tab,{}).get(col)
        if not li or not colonne: ignores.append((tab,rc,col)); continue
        ligne, _ = li; cible = wb[MAP_FEUILLES[tab]][colonne+str(ligne)]
        if _est_formule(cible):
            if isinstance(val,(int,float)) and not isinstance(val,bool): totaux.append((MAP_FEUILLES[tab], colonne+str(ligne), tab, rc, col, float(val)))
            continue
        cible.value = val; ecrits += 1; par[tab] = par.get(tab,0)+1
    for tab, lignes in dyn.items():
        debut, fin, col_lib = MAP_DYNAMIQUES[tab]; ws = wb[MAP_FEUILLES[tab]]; r = debut
        for item in lignes:
            if r > fin: ignores.append((tab,'ligne_'+str(r),'depassement')); break
            if col_lib and item.get('libelle'): ws[col_lib+str(r)] = item['libelle']
            for col, v in (item.get('valeurs') or {}).items():
                colonne = MAP_COLONNES.get(tab,{}).get(col)
                if colonne and v is not None:
                    cc = ws[colonne+str(r)]
                    if _est_formule(cc):
                        if isinstance(v,(int,float)) and not isinstance(v,bool): totaux.append((MAP_FEUILLES[tab], colonne+str(r), tab, 'ligne_'+str(r), col, float(v)))
                    else: cc.value = v; ecrits += 1; par[tab] = par.get(tab,0)+1
            r += 1
    return ecrits, totaux, ignores, par
def _comparer_et_annoter(chemin_calcule, wb, totaux, doc=None, tol=TOLERANCE_DA):
    calc = None
    if chemin_calcule:
        try: calc = openpyxl.load_workbook(chemin_calcule, data_only=True)
        except Exception: calc = None
    replis = {}
    if calc is None and doc:
        for f in (doc.get('controles') or {}).get('formules', []): replis[(f['tableau'],f['poste'],f['colonne'])] = f['valeur_recalculee']
    anomalies, arrondis = [], []
    for feuille, coord, tab, rc, col, extraite in totaux:
        if calc is not None:
            recalc = calc[feuille][coord].value
            if recalc is None: continue
            recalc = float(recalc)
        else:
            if (tab,rc,col) not in replis: continue
            recalc = float(replis[(tab,rc,col)])
        e = extraite - recalc
        if abs(e) <= 1e-9: continue
        c = wb[feuille][coord]
        if abs(e) <= tol:
            c.fill = _JAUNE_FOND
            c.comment = Comment('ECART D ARRONDI (+/- 1 DA)\nValeur lue: '+format(extraite,',.2f')+'\nValeur recalculee: '+format(recalc,',.2f')+'\nEcart: '+format(e,',.2f')+'\nSimple arrondi, aucune correction.', 'Controle extraction')
            c.comment.width = 320; c.comment.height = 150
            arrondis.append({'tableau':tab,'poste':rc,'colonne':col,'feuille':feuille,'cellule':coord,'valeur_extraite':extraite,'valeur_recalculee':recalc,'ecart':round(e,2)})
            continue
        c.border = _ROUGE; c.fill = _ROUGE_FOND; c.font = _ROUGE_TEXTE
        c.comment = Comment('INCOHERENCE DETECTEE\nValeur lue: '+format(extraite,',.2f')+'\nValeur recalculee: '+format(recalc,',.2f')+'\nEcart: '+format(e,',.2f')+'\nLa cellule affiche la valeur RECALCULEE.', 'Controle extraction')
        c.comment.width = 340; c.comment.height = 190
        anomalies.append({'tableau':tab,'poste':rc,'colonne':col,'feuille':feuille,'cellule':coord,'valeur_extraite':extraite,'valeur_recalculee':recalc,'ecart':round(e,2)})
    return anomalies, arrondis
def _feuille_archive(wb, doc, anomalies, arrondis, ignores):
    if '0. Données extraites' in wb.sheetnames: del wb['0. Données extraites']
    ws = wb.create_sheet('0. Données extraites', 0)
    ident = doc.get('identite') or {}; synth = (doc.get('controles') or {}).get('synthese', {})
    ws['A1'] = 'DONNEES EXTRAITES DE LA LIASSE — PIECE DE REFERENCE'; ws['A1'].font = Font(bold=True, size=14)
    ws['A2'] = 'Valeurs lues sur la liasse. Feuilles suivantes = copie de travail à totaux recalculés.'; ws['A2'].font = Font(italic=True, size=9)
    infos = [('Fichier source', doc.get('document',{}).get('fichier_source')), ('Entreprise', ident.get('entreprise')), ('NIF (15)', ident.get('nif_15')), ('Exercice clos', ident.get('exercice_clos')), ('N', ident.get('annee_n')), ('N-1', ident.get('annee_n1')), ('Dossier homogene', 'OUI' if ident.get('dossier_homogene') else 'NON'), ('Formules en ecart', synth.get('formules_en_ecart')), ('Controles en ecart', synth.get('controles_en_ecart')), ('Ecarts critiques', synth.get('ecarts_critiques'))]
    r = 4
    for lib, val in infos: ws.cell(r,1,lib).font = Font(bold=True); ws.cell(r,2,val); r += 1
    r += 1; ws.cell(r,1,'ECARTS SIGNIFICATIFS').font = Font(bold=True, size=12); r += 1
    for a in anomalies: ws.cell(r,1,a['tableau']); ws.cell(r,2,a['poste']); ws.cell(r,3,a['feuille']+'!'+a['cellule']); ws.cell(r,4,a['valeur_extraite']); ws.cell(r,5,a['valeur_recalculee']); ws.cell(r,6,a['ecart']); r += 1
    if not anomalies: ws.cell(r,2,'Aucun ecart significatif.'); r += 1
    r += 1; ws.cell(r,1,'ECARTS D ARRONDI').font = Font(bold=True, size=12); r += 1
    for a in arrondis: ws.cell(r,1,a['tableau']); ws.cell(r,2,a['poste']); ws.cell(r,3,a['feuille']+'!'+a['cellule']); ws.cell(r,4,a['ecart']); r += 1
    if ignores:
        r += 1; ws.cell(r,1,'POSTES NON REPORTES').font = Font(bold=True, size=12); r += 1
        for tab, rc, col in ignores: ws.cell(r,1,tab); ws.cell(r,2,rc); ws.cell(r,3,col); r += 1
    for col, w in zip('ABCDEFG', [26,46,20,24,20,20,16]): ws.column_dimensions[col].width = w
    return ws
def _ecrire_trace(wb, doc):
    pdf = (doc.get('document') or {}).get('fichier_source') or '?'
    for ws in wb.worksheets:
        if ws.title == '0. Données extraites': continue
        nums = []
        for p in doc.get('pages', []):
            for tab in (p.get('donnees') or {}):
                if MAP_FEUILLES.get(tab) == ws.title: nums.append(p['numero_page_scannee'])
        if nums:
            ws['H4'] = 'Source : ' + str(pdf); ws['H5'] = 'Page(s) scannee(s) : ' + ' / '.join(str(n) for n in sorted(set(nums)))
            for cc in ('H4','H5'): ws[cc].font = Font(italic=True, size=8, color='808080')
def _proteger_formules(wb):
    nl, nv = 0, 0
    for ws in wb.worksheets:
        for ligne in ws.iter_rows():
            for c in ligne:
                if isinstance(c.value, str) and c.value.startswith('='): c.protection = _VERROU; nv += 1
                else: c.protection = _LIBRE; nl += 1
        ws.protection.sheet = True
    return nl, nv
def transposer(doc, sortie_xlsx, modele=None, tol=TOLERANCE_DA):
    modele = Path(modele or MODELE_XLSX); sortie = Path(sortie_xlsx); tmp = sortie.with_suffix('.tmp.xlsx')
    shutil.copy(modele, tmp)
    wb = openpyxl.load_workbook(tmp)
    for ws in wb.worksheets: ws.protection.sheet = False
    ecrits, totaux, ignores, par = _ecrire_valeurs(wb, doc)
    _ecrire_trace(wb, doc)
    wb.save(tmp)
    recalcule = False
    if RECALC_DISPONIBLE:
        try: subprocess.run([sys.executable, str(RECALC_TOOL), str(tmp), '180'], capture_output=True, timeout=300); recalcule = True
        except Exception as e: log('Recalcul indisponible : ' + str(e))
    wb = openpyxl.load_workbook(tmp)
    anomalies, arrondis = _comparer_et_annoter(tmp if recalcule else None, wb, totaux, doc, tol)
    _feuille_archive(wb, doc, anomalies, arrondis, ignores)
    nl, nv = _proteger_formules(wb)
    wb.save(sortie); tmp.unlink(missing_ok=True)
    if RECALC_DISPONIBLE:
        try: subprocess.run([sys.executable, str(RECALC_TOOL), str(sortie), '180'], capture_output=True, timeout=300)
        except Exception as e: log('Recalcul final indisponible : ' + str(e))
    doc.setdefault('controles', {})['transposition'] = {'classeur':sortie.name,'valeurs_ecrites':ecrits,'totaux_compares':len(totaux),'ecarts_signales':len(anomalies),'ecarts_arrondis':len(arrondis),'postes_non_reportes':len(ignores),'cellules_verrouillees':nv,'cellules_saisissables':nl,'recalcul_libreoffice':RECALC_DISPONIBLE,'valeurs_par_tableau':par,'detail_ecarts':anomalies}
    return doc['controles']['transposition']
print('✅ Transposition OK')

In [ ]:
# PIPELINE : extraction 2 passes + pages blanches + progression + checkpoint
deja = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter = [p for p in pdfs if p.stem not in deja]
log('A traiter : ' + str(len(a_traiter)) + ' | déjà traités : ' + str(len(deja)))
t_total = time.time(); n_ok = n_err = 0; prompt_cache = {}
for num, pdf_path in enumerate(a_traiter, start=1):
    t0 = time.time(); prefix = '[' + str(num).zfill(4) + '/' + str(len(a_traiter)) + '] '
    log(prefix + 'DEMARRAGE ' + pdf_path.name)
    try:
        pages = pdf_to_pages(pdf_path)
        log(prefix + 'rendu ' + str(len(pages)) + ' pages en ' + str(round(time.time()-t0,1)) + 's')
        pobjs, actives = [], []
        for p in pages:
            (pobjs.append(build_blank_page(p, pdf_path.name)) if is_blank(p['image']) else actives.append(p))
        log(prefix + 'blanches ' + str(len(pobjs)) + ' | actives ' + str(len(actives)))
        ti = to = 0
        if actives:
            mini = [resize(p['image'], 600) for p in actives]; reps1 = []
            for bs in chunks(mini, CLASSIF_BATCH_SIZE): reps1 += ask_batch(PROMPT_CLASSIF, bs)
            ti += sum(r['tokens_in'] for r in reps1); to += sum(r['tokens_out'] for r in reps1)
            ptyped = []
            for p, rep in zip(actives, reps1):
                d = parse_json(rep['text']); types, nums = types_from_title(norm_str(d.get('titre')))
                if not types:
                    tm = (norm_str(d.get('type')) or '').upper()
                    types = [tm] if tm in ('ACTIF','PASSIF','TCR','DECL') else ['AUTRE']
                p['types'] = types; p['annexe_nums'] = nums; ptyped.append((p, tuple(types)))
            del mini, reps1; gc.collect()
            groups = {}
            for p, key in ptyped: groups.setdefault(key, []).append(p)
            extracted = {}
            for key, plist in groups.items():
                prompt = prompt_cache.get(key)
                if prompt is None:
                    prompt = build_prompt(list(key)) if key != ('AUTRE',) else ('Lis cette page. Extrais en JSON: type_page, titre_page, entete, tables.' + chr(10) + chr(10).join(RULES))
                    prompt_cache[key] = prompt
                for batch in chunks(plist, GPU_BATCH_SIZE):
                    reps = ask_batch(prompt, [p['image'] for p in batch])
                    for p, rep in zip(batch, reps):
                        ti += rep['tokens_in']; to += rep['tokens_out']
                        extracted[p['index']] = build_page_object(p, list(key), parse_json(rep['text']), rep, pdf_path.name)
                    gc.collect(); torch.cuda.empty_cache()
            for p in actives:
                pobjs.append(extracted.get(p['index'], build_page_object(p, ['AUTRE'], {}, None, pdf_path.name)))
        pobjs.sort(key=lambda x: x['pdf_page_index'])
        doc = build_document(pdf_path, pages, pobjs, ti, to, time.time()-t0)
        with open(JSON_DIR / (pdf_path.stem + '.json'), 'w', encoding='utf-8') as f: json.dump(doc, f, ensure_ascii=False, indent=2, default=str)
        n_ok += 1
        log(prefix + 'OK ' + pdf_path.name + ' | ' + str(round(time.time()-t0,1)) + 's | tok=' + str(ti+to))
    except Exception as e:
        n_err += 1; log(prefix + 'ERREUR ' + pdf_path.name + ' — ' + type(e).__name__ + ': ' + str(e)); continue
log('✅ Extraction terminée en ' + str(round(time.time()-t_total,1)) + 's | OK ' + str(n_ok) + ' | Erreurs ' + str(n_err))

In [ ]:
# POST-TRAITEMENT : identité + cohérence + transposition Excel + JSON contrôlé
def traiter_dossier(chemin_json, avec_excel=True):
    doc = json.loads(Path(chemin_json).read_text(encoding='utf-8'))
    doc = appliquer_controles(doc)
    stem = Path(chemin_json).stem
    if avec_excel and Path(MODELE_XLSX).exists():
        try: transposer(doc, XLSX_DIR / (stem + '.xlsx'))
        except Exception as e: log('❌ Transposition ' + stem + ' : ' + str(e))
    elif avec_excel: log('⚠️ Modèle introuvable : ' + str(MODELE_XLSX))
    sortie = CONTROLE_DIR / (stem + '.controle.json')
    sortie.write_text(json.dumps(doc, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    return doc, sortie
def resumer(doc):
    ident = doc.get('identite') or {}; s = (doc.get('controles') or {}).get('synthese', {}); t = (doc.get('controles') or {}).get('transposition', {})
    print('  ' + str(ident.get('entreprise') or '?') + ' | NIF ' + str(ident.get('nif_15') or '—') + ' | ' + str(ident.get('libelle_n') or 'N'))
    print('    homogene:', 'OUI' if ident.get('dossier_homogene') else 'NON', '| formules ecart:', s.get('formules_en_ecart'), '| controles ecart:', s.get('controles_en_ecart'), '| critiques:', s.get('ecarts_critiques'))
    if t: print('    Excel:', t.get('valeurs_ecrites'), 'valeurs |', t.get('ecarts_signales'), 'ecart(s) |', t.get('ecarts_arrondis'), 'arrondi(s)')
fichiers = sorted(JSON_DIR.glob('*.json'))
log('Post-traitement de ' + str(len(fichiers)) + ' dossier(s)')
for f in fichiers:
    try:
        doc, sortie = traiter_dossier(f); print('📄 ' + f.name); resumer(doc)
    except Exception as e: log('❌ ' + f.name + ' : ' + str(e))
print('✅ Post-traitement terminé | ' + str(CONTROLE_DIR) + ' | ' + str(XLSX_DIR))

In [ ]:
# TESTS HORS MODÈLE
_ok = _tot = 0
def _t(lib, obt, att):
    global _ok, _tot; _tot += 1; bon = obt == att; _ok += bon
    print(('✅' if bon else '❌') + ' ' + lib + ('' if bon else '  obtenu=' + repr(obt) + ' attendu=' + repr(att)))
_t('NIF conforme', nif_15('002119116225582'), ('002119116225582','CONFORME'))
_t('NIF court complete', nif_15('2119116225582'), ('002119116225582','COMPLETE'))
_t('Classification A3/A4', sorted(types_from_title('3/ Charges de personnel, impots, taxes et versements assimiles, autres services | 4/ Autres charges et produits operationnels')[0]), ['A3','A4'])
_t('Classification A9 (pas TCR)', types_from_title('9/ Tableau de determination du resultat fiscal | I. Resultat net de l exercice (Compte de resultat)')[0], ['A9'])
print(str(_ok) + '/' + str(_tot) + ' tests passés')